# VERİ SETİ

In [1]:
import torch
import kagglehub
import os
import re
import pandas as pd
import nltk

from nltk.tokenize import sent_tokenize
from collections import Counter

nltk.download("punkt")
nltk.download("punkt_tab")

# Dataseti indir
path = kagglehub.dataset_download("vivekmettu/wikitext2-data")

print("Dataset yolu:", path)
print("Dosyalar:", os.listdir(path))

# Train dosyasını oku
file_path = os.path.join(path, "train.txt")

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

df = pd.DataFrame(lines, columns=["text"])

print(df.shape)
df.head()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset yolu: /kaggle/input/datasets/vivekmettu/wikitext2-data
Dosyalar: ['test.txt', 'train.txt']
(37333, 1)
cuda


In [2]:
print(len(df))

37333


In [3]:
# Baştaki ve sondaki boşlukları sil
df["text"] = df["text"].str.strip()

# Boş satırları sil
df = df[df["text"] != ""]

# Tek başına kalan " satırlarını sil
df = df[df["text"] != '"']

# Wikipedia başlıklarını sil
df = df[
    ~df["text"].str.match(r"^\s*(=\s*)+.*?(\s*=)+\s*$", na=False)
]

# WikiText özel karakterlerini düzelt
df["text"] = df["text"].str.replace("@-@", "-", regex=False)
df["text"] = df["text"].str.replace("@.@", ".", regex=False)

print(df.shape)
df.head()

(17556, 1)


,text
3,The 2013 – 14 season was the <unk> season of c...
4,"Nigel Worthington , starting his first full se..."
5,35 players made at least one appearance in nat...
9,The 2012 – 13 season was York City 's first se...
10,Following the previous season 's conclusion Le...


In [4]:
cumleler = []

for text in df["text"]:
    cumleler.extend(sent_tokenize(text))

# Baştaki ve sondaki boşlukları sil
cumleler = [c.strip() for c in cumleler]

# Çok kısa cümleleri kaldır
cumleler = [c for c in cumleler if len(c.split()) >= 3]

print("Toplam cümle:", len(cumleler))

for i in range(10):
    print(cumleler[i])

Toplam cümle: 79999
The 2013 – 14 season was the <unk> season of competitive association football and 77th season in the Football League played by York City Football Club , a professional football club based in York , North Yorkshire , England .
Their 17th - place finish in 2012 – 13 meant it was their second consecutive season in League Two .
The season ran from 1 July 2013 to 30 June 2014 .
Nigel Worthington , starting his first full season as York manager , made eight permanent summer signings .
By the turn of the year York were only above the relegation zone on goal difference , before a 17 - match unbeaten run saw the team finish in seventh - place in the 24 - team 2013 – 14 Football League Two .
This meant York qualified for the play - offs , and they were eliminated in the semi - final by Fleetwood Town .
York were knocked out of the 2013 – 14 FA Cup , Football League Cup and Football League Trophy in their opening round matches .
35 players made at least one appearance in natio

In [5]:
kelimeler = []

for cumle in cumleler:
    kelimeler.extend(cumle.split())

vocab = sorted(set(kelimeler))

print("Toplam kelime:", len(kelimeler))
print("Benzersiz kelime:", len(vocab))

counter = Counter(kelimeler)
print(counter.most_common(50))

Toplam kelime: 2005867
Benzersiz kelime: 33222
[('the', 113006), (',', 99761), ('.', 76268), ('of', 56603), ('<unk>', 53815), ('and', 49808), ('in', 39382), ('to', 39132), ('a', 34225), ('""', 28265), ('was', 20985), ('The', 17503), ('-', 16974), ('that', 14134), ('as', 14011), ("'s", 13950), ('on', 13652), ('for', 13288), ('with', 12580), ('by', 12138), ('(', 11741), (')', 11741), ('is', 11633), ('from', 8931), ('his', 8394), ('at', 8176), ('were', 7329), ('it', 6748), ('he', 5947), ('an', 5894), ('had', 5698), ('which', 5542), ('In', 5490), ('be', 4787), ('are', 4668), (';', 4216), ('their', 4126), ('but', 4034), ('first', 3973), ('not', 3905), ('also', 3746), ('–', 3737), (':', 3721), ('its', 3696), ('or', 3634), ('her', 3485), ('have', 3442), ('one', 3349), ('has', 3321), ('been', 3257)]


In [6]:
kelimeler = []

for cumle in cumleler:
    kelimeler.extend(cumle.split())

vocab = sorted(set(kelimeler))

print("Toplam kelime:", len(kelimeler))
print("Benzersiz kelime:", len(vocab))

from collections import Counter

counter = Counter(kelimeler)

print(counter.most_common(50))

Toplam kelime: 2005867
Benzersiz kelime: 33222
[('the', 113006), (',', 99761), ('.', 76268), ('of', 56603), ('<unk>', 53815), ('and', 49808), ('in', 39382), ('to', 39132), ('a', 34225), ('""', 28265), ('was', 20985), ('The', 17503), ('-', 16974), ('that', 14134), ('as', 14011), ("'s", 13950), ('on', 13652), ('for', 13288), ('with', 12580), ('by', 12138), ('(', 11741), (')', 11741), ('is', 11633), ('from', 8931), ('his', 8394), ('at', 8176), ('were', 7329), ('it', 6748), ('he', 5947), ('an', 5894), ('had', 5698), ('which', 5542), ('In', 5490), ('be', 4787), ('are', 4668), (';', 4216), ('their', 4126), ('but', 4034), ('first', 3973), ('not', 3905), ('also', 3746), ('–', 3737), (':', 3721), ('its', 3696), ('or', 3634), ('her', 3485), ('have', 3442), ('one', 3349), ('has', 3321), ('been', 3257)]


In [7]:
# ====================================================
# =============== VOCABULARY OLUŞTURMA ===============
# ====================================================

ozel_tokenlar = [
    "<PAD>",
    "<SOS>",
    "<EOS>",
    "<UNK>"
]

vocab = ozel_tokenlar + vocab

word_to_ix = {kelime: i for i, kelime in enumerate(vocab)}

ix_to_word = {i: kelime for kelime, i in word_to_ix.items()}

print("Vocabulary Boyutu:", len(word_to_ix))
print()

for token in ozel_tokenlar:
    print(token, "->", word_to_ix[token])


Vocabulary Boyutu: 33226

<PAD> -> 0
<SOS> -> 1
<EOS> -> 2
<UNK> -> 3


In [8]:
# ====================================================
# ============= DataSet Sınıfı Tanımlama =============
# ====================================================

# Verilerimizi PyTorch modelinin anlayacağı Tensor formatına getiren
# ve her bir cümleyi Encoder/Decoder girdisi olarak parçalayan aşama.
import torch
from torch.utils.data import Dataset, DataLoader

class CumleDataset(Dataset):
  def __init__(self, dx, word_to_ix):
    self.df = dx  # Cümlelerimizin olduğu pandas serisi
    self.word_to_ix = word_to_ix  # Kelime-Sayı sözlüğümüz

  def __len__(self):
    # PyTorch veri setinde toplam kaç örnek olduğunu bilmek ister
    return len(self.df)

  def __getitem__(self, idx):
    # Veri setinden idx. sıradaki cümleyi çekip kelimelerine ayırıyoruz
    cumle = self.df[idx]
    tokens = cumle.split()

    # NOT: Burada kodunda üzerine yazma (overwrite) yapmışsın, son hali geçerli:
    # Encoder tam cümleyi sonuna <EOS> (Bitiş) alarak okur.
    encoder_input = tokens + ["<EOS>"]
    # Decoder cümleye başlarken önüne <SOS> (Başlangıç) alır.
    decoder_input = ["<SOS>"] + tokens
    # Target (Hedef/Etiket) ise modelin tahmin etmesini istediğimiz şeydir: Cümle + <EOS>
    target = tokens + ["<EOS>"]

    # Kelime listelerini sayısal ID listelerine dönüştürüyoruz. Sözlükte yoksa <UNK> veriyoruz.
    encoder_idleri = [self.word_to_ix.get(token, self.word_to_ix["<UNK>"]) for token in encoder_input]
    decoder_idleri = [self.word_to_ix.get(token, self.word_to_ix["<UNK>"]) for token in decoder_input]
    target_idleri = [self.word_to_ix.get(token, self.word_to_ix["<UNK>"]) for token in target]

    # PyTorch listelerle çalışamaz, bunları Long (Tam Sayı) Tensor'üne çeviriyoruz.
    encoder_idleri = torch.tensor(encoder_idleri, dtype=torch.long)
    decoder_idleri = torch.tensor(decoder_idleri, dtype=torch.long)
    target_idleri = torch.tensor(target_idleri, dtype=torch.long)

    return encoder_idleri, decoder_idleri, target_idleri

# İlk ham dataset nesnemizi oluşturuyoruz
dataset = CumleDataset(cumleler, word_to_ix)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [9]:
from collections import Counter

counter = Counter(kelimeler)

print("Toplam vocab:", len(counter))
print("1 kez geçen:", sum(v == 1 for v in counter.values()))
print("2 kez geçen:", sum(v == 2 for v in counter.values()))
print("3 kez geçen:", sum(v == 3 for v in counter.values()))

Toplam vocab: 33222
1 kez geçen: 65
2 kez geçen: 197
3 kez geçen: 5967


In [10]:
# ==========================================================================
# ======= DataLoader ve Collate Function (Dinamik Dolgu ve Maskeler) =======
# ==========================================================================

# Grup grup (Batch) veri yüklerken cümlelerin boyları eşit olmalıdır. collate_fn
# fonksiyonu kısa cümlelerin sonuna <PAD> ekler ve modelin geleceği görmesini engelleyen maskeleri üretir.
def collate_fn(batch):
  # batch listesini çözer: [(enc1, dec1, tgt1), (enc2, dec2, tgt2), ...] -> enc_batch, dec_batch...
  enc_batch, dec_batch, tgt_batch = zip(*batch)

  PAD = word_to_ix["<PAD>"]
  SOS = word_to_ix["<SOS>"]

  # Bu batch (grup) içindeki en uzun listeleri buluyoruz
  max_enc_len = max([len(seq) for seq in enc_batch])
  max_dec_len = max([len(seq) for seq in dec_batch])
  max_tgt_len = max([len(seq) for seq in tgt_batch])
  # Hepsini tek bir ortak maksimum uzunluğa eşitlemek için en büyüğünü seçiyoruz
  max_len = max(max_enc_len, max_dec_len, max_tgt_len)

  # Verilen listeyi max_len olana kadar sonuna PAD (0) ekleyerek uzatan iç fonksiyon
  def pad(seq, max_len):
    return seq + [PAD] * (max_len - len(seq))

  # Tüm batch elemanlarını listeye çevirip (tolist) pad fonksiyonuyla eşitliyoruz
  enc_batch = [pad(seq.tolist(), max_len) for seq in enc_batch]
  dec_batch = [pad(seq.tolist(), max_len) for seq in dec_batch]
  tgt_batch = [pad(seq.tolist(), max_len) for seq in tgt_batch]

  # Matrix işlemleri için Tensor formuna getiriyoruz
  enc_batch = torch.tensor(enc_batch, dtype=torch.long)
  dec_batch = torch.tensor(dec_batch, dtype=torch.long)
  tgt_batch = torch.tensor(tgt_batch, dtype=torch.long)

  # ===--- MASKELERİN OLUŞTURULMASI ---===
  # Causal Mask (Üçgen Maske): Decoder'ın 2. kelimeyi tahmin ederken 3. ve 4. kelimeleri görmesini engeller.
  causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1).bool()

  # Padding Maskeleri: Modelin <PAD> (boşluk) tokenlarına odaklanıp vakit kaybetmesini engeller.
  dec_pad_mask = (dec_batch == PAD)
  enc_pad_mask = (enc_batch == PAD)

  return (enc_batch, dec_batch, tgt_batch, causal_mask, dec_pad_mask, enc_pad_mask)

# DataLoader'ı tanımlıyoruz. Batch size'ı şimdilik 16 yapalım
# Her epoch başında cümlelerin sırasını karıştırmak için shuffle=True yapıyoruz ve collate_fn ile işliyoz.
train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

In [11]:
# ========================================================
# ============== POSITIONAL ENCODING =====================
# ========================================================

# Transformer modelleri kelimeleri sırayla değil, hepsini aynı anda (paralel) içeri alır
# Bu yüzden hangi kelimenin başta, hangisinin sonda olduğunu bilmezler. Bu sınıf,
# kelime vektörlerine "pozisyon/sıra" bilgisi ekler.

import math
from torch import nn

class PositionalEncoding(nn.Module):
  def __init__(self, embedding_dim, max_len=5000):
      super().__init__()

      # Boş bir konum matrisi oluştur (Satır: Maksimum uzunluk, Sütun: Vektör boyutu)
      pe = torch.zeros(max_len, embedding_dim)

      # 0'dan max_len'e kadar olan sayıları dik bir vektör yapar (max_len, 1)
      position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

      # Formüldeki alt paydayı (frekans değerini) hesaplar
      div_term = torch.exp(
          torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))

      # Çift indeksli sütunlara (0, 2, 4...) Sinüs dalgası yerleştirir
      pe[:, 0::2] = torch.sin(position * div_term)

      # Tek indeksli sütunlara (1, 3, 5...) Kosinüs dalgası yerleştirir
      pe[:, 1::2] = torch.cos(position * div_term)

      # Batch boyutu için başa bir boyut ekler -> (1, max_len, embedding_dim)
      pe = pe.unsqueeze(0)

      # register_buffer: Bu matrisin modelin güncelleyeceği bir "ağırlık" olmadığını,
      # sabit bir değer olduğunu ama modelle birlikte taşınması gerektiğini söyler.
      self.register_buffer("pe", pe)

  def forward(self, x):
      # x'in boyutu: (batch, seq_len, embedding_dim)
      seq_len = x.size(1)

      # Ham kelime vektörlerinin (x) üzerine, o anki uzunluk kadar konum bilgisi ekleyip geri döndürür.
      x = x + self.pe[:, :seq_len]
      return x

In [12]:
# ======================================================================= #
# ================== ENCODER BLOCK (ANLAYICI KATMAN)===================== #
# ======================================================================= #

# Girdi cümlesindeki kelimelerin birbirleriyle olan ilişkilerini (Self-Attention) çözen
# ve bunu rafine eden blok.

class EncoderBlock(nn.Module):
  def __init__(self, embedding_dim, num_heads):
    super().__init__()

    # Multi-head Attention: Kelimelerin birbiriyle eşleşme skorunu bulur.
    # batch_first=True girdilerin (batch, seq, feature) düzeninde olduğunu belirtir.
    self.enc_attention = nn.MultiheadAttention(embedding_dim, num_heads=num_heads, batch_first=True)

    # LayerNorm: Sayısal değerleri stabilize ederek modelin daha hızlı ve düzgün öğrenmesini sağlar.
    self.enc_norm1 = nn.LayerNorm(embedding_dim)

    # Feed Forward Network (FFN): Attention çıkışını alıp daha derin özellikler çıkaran doğrusal katmanlar.
    # Genelde boyutu 4 katına (embedding_dim * 4) çıkarıp sonra tekrar eski haline getirir.
    self.enc_ffn = nn.Sequential(
        nn.Linear(embedding_dim, embedding_dim * 4),
        nn.ReLU(),  # Negatif değerleri sıfırlayan aktivasyon fonksiyonu
        nn.Linear(embedding_dim * 4, embedding_dim)
    )
    self.enc_norm2 = nn.LayerNorm(embedding_dim)

    # Dropout
    self.dropout1 = nn.Dropout(0.1)
    self.dropout2 = nn.Dropout(0.1)

  def forward(self, x, enc_pad_mask):
    # 1. Aşama: Self-Attention. Query, Key ve Value değerlerinin üçü de x'ten gelir (Kelimeler kendilerine bakar).
    # key_padding_mask=enc_pad_mask sayesinde <PAD> tokenlarına bakılması engellenir.
    enc_out, _ = self.enc_attention(query=x, key=x, value=x, key_padding_mask=enc_pad_mask)

    # Residual Connection (x + enc_out): Orijinal bilgiyi kaybetmemek için girdiyi çıktıya ekliyoruz, sonra normalize ediyoruz.
    enc_out = self.enc_norm1(x + self.dropout1(enc_out))

    # 2. Aşama: Bilgiyi karıştırıp derinleştiren sinir ağı katmanı (FFN)
    ffn_out = self.enc_ffn(enc_out)

    # Yine Residual Connection ve norm işlemi.
    enc_out = self.enc_norm2(enc_out + self.dropout2(ffn_out))
    return enc_out

In [13]:
# ================================================================== #
# ========================= DECODER BLOCK ========================== #
# =================================================================== #

# Hem o ana kadar ürettiği kelimelere bakan (Masked Self-Attention) hem de
# Encoder'dan gelen kütüphane bilgilerini sorgulayan (Cross-Attention) bloktur.
class DecoderBlock(nn.Module):
  def __init__(self, embedding_dim, num_heads):
    super().__init__()

    # 1. Maskeli Attention: Decoder'ın kendi ürettiği geçmiş kelimelere bakması için.
    self.dec_attention = nn.MultiheadAttention(embedding_dim, num_heads=num_heads, batch_first=True)
    self.dec_norm1 = nn.LayerNorm(embedding_dim)

    # 2. Cross Attention: Encoder'ın çıktısını (kütüphaneyi) sorgulamak için.
    self.cross_attention = nn.MultiheadAttention(embedding_dim, num_heads=num_heads, batch_first=True)
    self.dec_norm2 = nn.LayerNorm(embedding_dim)

    # 3. Klasik Feed Forward katmanı
    self.dec_ffn = nn.Sequential(nn.Linear(embedding_dim, embedding_dim*4),
                                 nn.ReLU(),
                                 nn.Linear(embedding_dim * 4, embedding_dim))
    self.dec_norm3 = nn.LayerNorm(embedding_dim)

    # Dropout
    self.dropout1 = nn.Dropout(0.1)
    self.dropout2 = nn.Dropout(0.1)
    self.dropout3 = nn.Dropout(0.1)

  def forward(self, dec_emb, enc_out, causal_mask, dec_pad_mask, enc_pad_mask):
    # Adım 1: Masked Self Attention.
    # nn.MultiHeadAttention iki şey döndürür: 1. Attention çıktısı, 2. Attention ağırlıkları
    # ve bizim ihtiyacımız olan şey 1. çıktı. Peki 2. çıktı ne zaman kullanılır? Modeli analiz ederken
    # model "..." kelimesini çözerken hangi kelimelere baktı? dersin. O zaman heatmap çizerek vs.
    # attention görselleştirmesi yapabiliriz
    attn_out, _ = self.dec_attention(query=dec_emb, key=dec_emb, value=dec_emb, attn_mask=causal_mask, key_padding_mask=dec_pad_mask)
    res_out1 = self.dec_norm1(dec_emb + self.dropout1(attn_out))

    # Adım 2: Cross Attention (En Kritik Nokta!)
    # Query: Decoder'ın o anki durumu (res_out1)
    # Key ve Value: Encoder'ın ürettiği anlam kütüphanesi (enc_out)
    # Yani model "Elimdeki bu ipucuyla, encoder kütüphanesinde neye baksam doğru olur?" diye sorar.
    cross_attn_out, _ = self.cross_attention(query=res_out1, key=enc_out, value=enc_out, key_padding_mask=enc_pad_mask)
    res_out2 = self.dec_norm2(res_out1 + self.dropout2(cross_attn_out))

    # Adım 3: Son katman doğrusal işlemler ve çıkış
    ffn_out = self.dec_ffn(res_out2)
    res_out3 = self.dec_norm3(res_out2 + self.dropout3(ffn_out))

    return res_out3

In [14]:
# =============================================================================
# ============= TÜM PARÇALARI BİRLEŞTİREN ANA TRANSFORMER SINIFI ==============
# =============================================================================

# Burada kurduğumuz tüm küçük makineleri (Embedding, PE, Encoder Stack, Decoder Stack
# ve Çıkış Katmanı tek bir çatı altında topluyoruz.
import torch
import torch.nn as nn
embedding_dim = 64
num_heads = 8

class Transformerim(nn.Module):
  def __init__(self, vocab_size, embedding_dim, num_heads):
    super().__init__()

    # nn.Embedding: Sayısal ID'leri (örn: 5) alıp 64 boyutlu öğrenilebilir vektörlere çevirir.
    # padding_idx=0 vererek, 0 numaralı <PAD> tokenının hep sıfır kalmasını ve öğrenilmemesini sağlıyoruz.
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

    # Konum kodlayıcımız
    self.positional_encoding = PositionalEncoding(embedding_dim)

    # ModuleList kullanarak 6 adet Encoder bloğunu peş peşe (stack) diziyoruz.
    self.encoder = nn.ModuleList([EncoderBlock(embedding_dim, num_heads) for _ in range(6)])
    # 6 adet Decoder bloğunu peş peşe diziyoruz.
    self.decoder = nn.ModuleList([DecoderBlock(embedding_dim, num_heads) for _ in range(6)])

    # Son Çıkış Katmanı: 64 boyutlu gizli vektörü, sözlüğümüzdeki kelime sayısı (vocab_size) kadar skora (logits) dönüştürür.
    self.linear = nn.Linear(embedding_dim, vocab_size)

    # self._init_weights()

    # Xavier initialization
  # def _init_weights(self):
    # for m in self.modules():
       # if isinstance(m, nn.Linear):
        # nn.init.xavier_uniform_(m.weight)
          # if m.bias is not None:
            # nn.init.zeros_(m.bias)

  def forward(self, girdi_idleri, decoder_idleri, causal_mask, dec_pad_mask, enc_pad_mask):
    # --- ENCODER AKIŞI ---
    enc_emb = self.embedding(girdi_idleri) * math.sqrt(self.embedding.embedding_dim)     # Kelimeleri vektör yap
    enc_emb = self.positional_encoding(enc_emb)  # Pozisyon ekle
    for katman in self.encoder:
        enc_emb = katman(enc_emb, enc_pad_mask)  # 6 katmandan sırayla geçir (Kütüphane hazırlandı)

    # --- DECODER AKIŞI ---
    dec_emb = self.embedding(decoder_idleri)    * math.sqrt(self.embedding.embedding_dim)  # Hedef kelimeleri vektör yap
    dec_emb = self.positional_encoding(dec_emb)  # Pozisyon ekle
    for katman in self.decoder:
        # Her decoder katmanına hazır kütüphaneyi (enc_emb) de paslıyoruz.
        dec_emb = katman(dec_emb, enc_emb, causal_mask, dec_pad_mask, enc_pad_mask)

    # Vektörleri kelime skorlarına çeviriyoruz.
    # NOT: Softmax eklemiyoruz çünkü PyTorch'un CrossEntropyLoss fonksiyonu bunu içeride otomatik yapıyor.
    logits = self.linear(dec_emb)
    return logits

# Modelimizi ayağa kaldırıyoruz
model = Transformerim(vocab_size=len(word_to_ix), embedding_dim=embedding_dim, num_heads=num_heads).to(device)
torch.backends.cudnn.benchmark = True
# print("Model başarıyla oluşturuldu! Katmanlar: \n",model)

"The encoder is composed of a stack of N = 6 identical layers." ve "The decoder is also composed of a stack of N = 6 identical layers."

ve bu stackler uç uca ekleniyolar. Şu şekilde: input embedding -> encoder block 1 -> encoder block 2 -> encoder block 3 -> encoder block 4 -> encoder block 5 -> encoder block 6

Yani block 2, block 1 in çıktısını alıyor ve o şekilde devam ediyor.
Peki neden?
Şöyle düşünelim.

İlk blokta model belki şunu öğrendi:

kral  <----->  sarayda

İkinci blok artık ham embedding'e bakmıyor.
Onun yerine şuna bakıyor:

"Birinci blok zaten 'kral'ın sarayla ilişkili olduğunu çıkarmış. Ben bunun üstüne daha karmaşık ne öğrenebilirim?"

Peki paralel çalışmıyorlar mı?
Hayır.
Ama dikkat:

Bir bloğun içindeki attention hesapları GPU'da paralel yapılır.
Fakat

Block1 ->
Block2 ->
Block3
sıralıdır.

Çünkü Block2'nin çalışabilmesi için önce Block1'in bitmesi gerekir.

Encoder bir kütüphane hazırlıyor.
Decoder ise araştırmacı.
İlk decoder bloğu:
"Bana kütüphaneden kuş ile ilgili bilgileri getir."

İkinci decoder bloğu:
"Tamam, şimdi aynı kütüphaneden biraz daha ayrıntılı bilgi getir."

Üçüncü decoder bloğu:
"Şimdi aynı kütüphaneden başka açıdan bilgi getir."

Ama kütüphane değişmiyor.
Sadece araştırmacının soruları giderek daha kaliteli oluyor.
Yani:
Query değişiyor.
Key ve Value sabit kalıyor.
İşte Cross-Attention'ın özü bu.

In [15]:
# ===========================================================================
# ======================= LOSS VE OPTIMIZER ===============================
# ===========================================================================

criterion = nn.CrossEntropyLoss(ignore_index=word_to_ix["<PAD>"])

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

# ===========================================================================
# ======================== CHECKPOINT ================================
# ===========================================================================

import os

checkpoint_path = "/kaggle/working/transformer_last.pt"

start_epoch = 0
best_loss = float("inf")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_loss = checkpoint["best_loss"]

    print("Checkpoint yüklendi.")
else:
    print("Eğitim sıfırdan başlayacak.")

# ===========================================================================
# =========================== TRAINING LOOP ===============================
# ===========================================================================

epochs = 2

for epoch in range(epochs):

    epoch_loss = 0

    for batch_idx, (enc_batch,dec_batch,tgt_batch,causal_mask,dec_pad_mask,
                    enc_pad_mask) in enumerate(train_loader):

        enc_batch = enc_batch.to(device)
        dec_batch = dec_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        causal_mask = causal_mask.to(device)
        dec_pad_mask = dec_pad_mask.to(device)
        enc_pad_mask = enc_pad_mask.to(device)

        optimizer.zero_grad()

        logits = model(enc_batch,dec_batch,causal_mask,dec_pad_mask,enc_pad_mask)

        loss = criterion(logits.reshape(-1, len(word_to_ix)),tgt_batch.view(-1))

        epoch_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

        optimizer.step()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch} | "f"Batch {batch_idx}/{len(train_loader)} | "f"Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch} Ortalama Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:

        best_loss = avg_loss

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_loss": best_loss
        }

        torch.save(checkpoint, checkpoint_path)

        print("Yeni en iyi model kaydedildi.")

Eğitim sıfırdan başlayacak.
Epoch 0 | Batch 0/5000 | Loss: 10.6343
Epoch 0 | Batch 100/5000 | Loss: 6.9572
Epoch 0 | Batch 200/5000 | Loss: 6.8192
Epoch 0 | Batch 300/5000 | Loss: 6.2003
Epoch 0 | Batch 400/5000 | Loss: 6.5326
Epoch 0 | Batch 500/5000 | Loss: 6.2251
Epoch 0 | Batch 600/5000 | Loss: 6.3393
Epoch 0 | Batch 700/5000 | Loss: 5.7831
Epoch 0 | Batch 800/5000 | Loss: 6.2602
Epoch 0 | Batch 900/5000 | Loss: 5.7754
Epoch 0 | Batch 1000/5000 | Loss: 5.7862
Epoch 0 | Batch 1100/5000 | Loss: 5.6680
Epoch 0 | Batch 1200/5000 | Loss: 5.8653
Epoch 0 | Batch 1300/5000 | Loss: 5.8594
Epoch 0 | Batch 1400/5000 | Loss: 5.7141
Epoch 0 | Batch 1500/5000 | Loss: 5.3302
Epoch 0 | Batch 1600/5000 | Loss: 5.5538
Epoch 0 | Batch 1700/5000 | Loss: 5.5209
Epoch 0 | Batch 1800/5000 | Loss: 5.5427
Epoch 0 | Batch 1900/5000 | Loss: 5.3061
Epoch 0 | Batch 2000/5000 | Loss: 5.5121
Epoch 0 | Batch 2100/5000 | Loss: 5.2078
Epoch 0 | Batch 2200/5000 | Loss: 5.0859
Epoch 0 | Batch 2300/5000 | Loss: 5.1951

In [16]:
import torch

checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_loss": best_loss
}

torch.save(checkpoint, "/kaggle/working/transformer_last.pt")

print("Model kaydedildi.")

Model kaydedildi.


In [17]:
import os

print(os.path.exists("/kaggle/working/transformer_last.pt"))

True


In [18]:
# ============================================================================ #
# ===== INFERENCE (AUTOREGRESSIVE(KENDİ KENDİNİ BESLEYEN) CÜMLE ÜRETİMİ) ===== #
# ============================================================================ #

# Modelin gerçek hayatta nasıl çalıştığının simülasyonu. Model sadece <SOS> alır
# ve kendi ürettiği kelimeyi bir sonraki adıma girdi olarak vererek cümle tamamlayana kadar (<EOS>) döner.


model.eval()

# 1. Encoder'a ipucu kelimeleri veriyoruz: "tilki ormanda" (Model bunun anlamını çözecek)
enc_girdi = torch.tensor([[word_to_ix["The"], word_to_ix["king"]]], dtype=torch.long, device=device)

# 2. Decoder'ı sadece başlangıç tokenı olan <SOS> ile tetikliyoruz. Boyut: [1, 1] (Batch=1, Uzunluk=1)
dec_girdi = torch.tensor([[word_to_ix["<SOS>"]]], dtype=torch.long, device=device)

print("Üretim Başlıyor...\n")

with torch.no_grad():
  for i in range(20): # Maksimum 20 kelimeye kadar üretsin (Sonsuz döngü engeli)
    seq_len = dec_girdi.size(1)

    # O anki decoder uzunluğuna göre dinamik maskeleri hazırlıyoruz
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    dec_pad_mask = (dec_girdi == word_to_ix["<PAD>"]).to(device)
    enc_pad_mask = (enc_girdi == word_to_ix["<PAD>"]).to(device)

    # Modeli çalıştırıp tahmin skorlarını alıyoruz
    logits = model(enc_girdi, dec_girdi, causal_mask, dec_pad_mask, enc_pad_mask)

    # Logits boyutu: [1, seq_len, 54]. Biz sadece EN SON üretilen kelimenin tahmin skorlarına bakıyoruz.
    son_kelime_logits = logits[0, -1, :]

    # En yüksek olasılıklı kelime ID'sini buluyoruz
    temperature = 0.8
    top_k = 20

    # Temperature uygula
    logits = son_kelime_logits / temperature

    repetition_penalty = 1.3

    for token_id in dec_girdi[0]:
      logits[token_id] /= repetition_penalty

    # En yüksek top_k logiti seç
    topk_logits, topk_indices = torch.topk(logits, top_k)

    # Sadece bu 20 kelime üzerinde olasılık hesapla
    probs = torch.softmax(topk_logits, dim=-1)

    # Bu 20 kelime arasından rastgele seç
    sampled = torch.multinomial(probs, 1)

    # Gerçek vocabulary indexine geri dön
    en_yuksek_id = topk_indices[sampled].item()

    tahmin_kelime = ix_to_word[en_yuksek_id]

    # Adım adım süreci ekrana basıyoruz
    print(f"Adım {i+1}: Decoder Geçmişi: {[ix_to_word[idx.item()] for idx in dec_girdi[0]]} -> Tahmin: '{tahmin_kelime}'")

    # AUTOREGRESSIVE ÖZÜ: Tahmin edilen kelimeyi yeni bir tensor yapıp eski geçmişin sonuna ekliyoruz (cat).
    # Böylece bir sonraki döngüde model kendi ürettiği kelimeyi girdi olarak görecek.
    yeni_kelime_tensor = torch.tensor([[en_yuksek_id]], dtype=torch.long, device=device)
    dec_girdi = torch.cat([dec_girdi, yeni_kelime_tensor], dim=1)

    # Eğer model cümle bitti (<EOS>) dediyse döngüden hemen çık
    if en_yuksek_id == word_to_ix["<EOS>"]:
        break

# --- 3. TEMİZ ÇIKTI YAZDIRMA ---
# Özel tokenları (<SOS>, <EOS>) temizleyip insan gibi okuyalım
kelimeler = []
for idx in dec_girdi[0]:
    kelime = ix_to_word[idx.item()]
    if kelime in ["<SOS>", "<EOS>"]:
        continue
    kelimeler.append(kelime)

print("\nFinal Cümlesi:", " ".join(kelimeler))

Üretim Başlıyor...

Adım 1: Decoder Geçmişi: ['<SOS>'] -> Tahmin: 'The'
Adım 2: Decoder Geçmişi: ['<SOS>', 'The'] -> Tahmin: 'The'
Adım 3: Decoder Geçmişi: ['<SOS>', 'The', 'The'] -> Tahmin: 'list'
Adım 4: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list'] -> Tahmin: 'The'
Adım 5: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The'] -> Tahmin: '4'
Adım 6: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4'] -> Tahmin: '5'
Adım 7: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4', '5'] -> Tahmin: 'also'
Adım 8: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4', '5', 'also'] -> Tahmin: 'included'
Adım 9: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4', '5', 'also', 'included'] -> Tahmin: 'The'
Adım 10: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4', '5', 'also', 'included', 'The'] -> Tahmin: 'New'
Adım 11: Decoder Geçmişi: ['<SOS>', 'The', 'The', 'list', 'The', '4', '5', 'also', 'included', 'The', 'New'] -> Tahmin: 'York'
Adım 12: Dec

In [19]:
# ============================================================================ #
# =========== BELİRLİ BİR PREFİX (ÖN EK) İLE KELİME TAHMİN TESTİ ============= #
# ============================================================================ #

# Bu hücrede modele bir kelime verip, decoder kısmına da başlangıç kalıbı vererek
# bir sonraki kelimeyi tamamlama yeteneğini test ediyoruz.

model.eval()

# Encoder girdisi: "adam" (Hakkında konuşulan özne)
current_enc_input = torch.tensor([[word_to_ix["king"]]], dtype=torch.long, device=device)

# Decoder girdisi (Prefix): "<SOS> ofiste"
# Modele diyoruz ki: "Özne adam, mekân ofiste. Sence bir sonraki kelime ne olmalı?"
current_dec_input = torch.tensor([[word_to_ix["<SOS>"], word_to_ix["lives"]]], dtype=torch.long, device=device)

with torch.no_grad():
  # Güncel uzunluklara göre maskelerin ayarlanması
  seq_len_dec = current_dec_input.size(1)
  seq_len_enc = current_enc_input.size(1)

  causal_mask = torch.triu(torch.ones(seq_len_dec, seq_len_dec, device=device), diagonal=1).bool()
  dec_pad_mask = (current_dec_input == word_to_ix["<PAD>"]).to(device)
  enc_pad_mask = (current_enc_input == word_to_ix["<PAD>"]).to(device)

  # ID'leri modele fırlatıyoruz
  logits = model(current_enc_input, current_dec_input, causal_mask, dec_pad_mask, enc_pad_mask)

  # En son token olan "ofiste" kelimesinden sonra gelecek kelimenin skorlarını alıyoruz.
  son_kelime_logits = logits[0, -1, :]
  en_yuksek_id = son_kelime_logits.argmax().item()

  # Sayısal ID'yi metne geri çeviriyoruz
  tahmin_edilen_kelime = ix_to_word[en_yuksek_id]

# Sonuçları ekrana yazdırma
print(f"Encoder Girdi: {[ix_to_word[i.item()] for i in current_enc_input[0]]}")
print(f"Decoder Girdi (Prefix): {[ix_to_word[i.item()] for i in current_dec_input[0]]}")
print(f"Modelin Tahmini (Bir sonraki kelime): {tahmin_edilen_kelime}")

Encoder Girdi: ['king']
Decoder Girdi (Prefix): ['<SOS>', 'lives']
Modelin Tahmini (Bir sonraki kelime): historian
